---

<span style="font-size:24px"> <b> Deep learning to automate parameter extraction and model fitting of two-dimensional transistors: a jupyter notebook tutorial </b> </span>

---

This notebook will walk you through the approach we use in our paper, "Deep Learning to Automate Parameter Extraction and Model Fitting of Two-Dimensional Transistors" [Research 2026](https://spj.science.org/doi/10.34133/research.1103), also available on [arXiv](https://arxiv.org/abs/2507.05134), to train and test a neural network for inverse design. 

This notebook assumes you have already installed all of the packages specified in requirements.txt and that you've installed the transistor_extract module using pip. If you haven't done those things yet, follow the instructions [in the repository's README](https://github.com/RKABennett/deep-learning-for-transistor-parameter-extraction/blob/main/README.md) to get started. We provide some background information as we walk through the notebook, and refer you to specific portions of our paper for extra reading as we go.

<span style="font-size:20px"> <b> Overview of our pipeline </b> </span>

We guide the training of our inverse neural network using a secondary neural network that mimics a physics-based transistor simulator. We refer to this secondary neural network as the <i> forward neural network </i>. When we train the inverse neural network, we feed its output into the forward neural network to train it as a tandem neural network, as shown below.

<img src="images/tandem.png" alt="Drawing" style="width: 90%;"/>

Our full pipeline consists of five steps:

1. <b> Data generation: </b> Use a comprehensive physics-based model (e.g., Sentaurus Device) to generate a training data set of input parameters and calculated electrical characteristics. (In this notebook, we provide the simulated data, which we will pre-process.)
2. <b> Training a surrogate model: </b> Train an intermediate forward neural network to approximate the physics-based model.
3. <b> Pretraining the inverse neural network: </b> Generate a much larger augmented training set using the forward neural network and use this augmented data to pre-train the inverse neural network, calling the forward neural network repeatedly to guide training as a tandem neural network.
4. <b> Fine-tuning the inverse neural network: </b> Finish training the inverse neural network using the original data set from step 1, once again calling the forward neural network repeatedly to guide training as a tandem neural network.
5. <b> Testing the inverse neural network: </b>  Test the inverse neural network by inputting new electrical data, extracting the model parameters, and calling the physics-based model on the predicted model parameters to assess the accuracy of the extraction. (In this notebook, we use a surrogate model in place of the full physics-based model.)

The data we are using has already been generated for us using Sentaurus Device, an industry-standard technology computer-aided design (TCAD) solver, but it's not in an easy-to-use format by default, so our first real task is process our data into numpy arrays. But before that, let's import all the required packages and define functions that will help along the way.

In [ ]:
%matplotlib inline
import glob
import json
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import random
import scipy.stats
import sys
from tensorflow.keras.models import load_model

import numpy as np
import tensorflow as tf  
import transistor_extract as TE


working_dir = Path().resolve()
repo_root = working_dir.parent
root_dir = working_dir.parent
working_dir = str(working_dir)
repo_root = Path(str(repo_root))
root_dir = str(root_dir)

<span style="font-size:20px"> <b> Frequently used variables are defined in config.json </b> </span>

There are a few variables that we often wish to keep consistent across experiments, such as training rates. These key variables are defined in the root of our GitHub directory in config.json. This way, when variables are loaded in several scripts, changing the variable in config.json is all you need to do to change it in every script. Below, we import the config.json file, which we call from frequently throughout this notebook.

In [ ]:
config_path = os.path.join(root_dir, "config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

<span style="font-size:20px"> <b>  Processing our data</b> </span>

Our inverse neural network accepts current vs. gate-to-source voltage (I<sub>d</sub>-V<sub>gs</sub>) curves and their derivatives, at V<sub>ds</sub> = 0.1 and 1 V, in both linear and logarithmic space, and aims to output a set of model parameters that will allow Sentaurus Device TCAD to reproduce these I<sub>d</sub>-V<sub>gs</sub> curves. We obtained these data by running Sentaurus Device TCAD simulations on the two-dimensional transistor geometry shown below. All devices are back-gated with 500 nm channel lengths and 100 nm SiO<sub>2</sub> back gates. Our full data set consists of data from 26,000 devices. 

<div style="text-align: center;">
  <img src="images/transistor.png" alt="Drawing" style="width: 30%;"/>
</div>


We format our I<sub>d</sub>-V<sub>gs</sub> data as a matrix, with relevant features arranged as columns, as shown below. For justification as to why we include all of these features, see Supplementary Section 3 of our paper.

<div style="text-align: center;">
  <img src="images/input_matrix.png" alt="Drawing" style="width: 50%;"/>
</div>

Our output data is the eight model parameters that we're going to treat as fitting parameters. Each of the parameters is allowed to vary across a specified range (which we decided when we generated our training set using Sentaurus). We list the parameters below, and refer you to the discussion in our paper around Table I and in Supplementary Section S2 for the physical meaning of each parameter and for justifications of why we're using the chosen ranges.

<div style="text-align: center;">
  <img src="images/params.png" alt="Drawing" style="width: 60%;"/>
</div>

Now we're going to load our training and test sets. Our raw data is in data/raw as textfiles. We're going to process these text files into numpy arrays, which we'll save in a new folder, data/processed. We use a built-in function in the transistor-extract library for this, which handles all of our pre-processing steps. See the paper for the particulars of how we preprocess our data. We'll save the input data array as X and output data as Y.

In [ ]:
np.random.seed(19700101)
random.seed(19700101)

raw_data_loc = str(repo_root / 'data' / 'raw')
processed_data_loc = str(repo_root / 'data' / 'processed')
os.makedirs(processed_data_loc, exist_ok=True)

Xscaling = np.loadtxt(Path(processed_data_loc) / 'Xscaling.dat')
Yscaling = np.loadtxt(Path(processed_data_loc) / 'Yscaling.dat')

X_array = [] # we're going to populate this with the IdVg data and derivatives
Y_array = [] # we're going to populate this with the relevant features

V = np.linspace(
    cfg["data"]["Vmin"],
    cfg["data"]["Vmax"],
    cfg["data"]["n_points"],
    )

X, Y = TE.process_folder(
                         raw_data_loc,
                         working_dir,
                         processed_data_loc,
                         V,
                         cfg["data"]["n_points"],
                         cfg["data"]["num_IdVg"],
                         cfg["data"]["num_feats"],
                         cfg["data"]["minval"],
                         print_progress = True, # change this to False to suppress printing
                         print_frequency = 1000
                         )


Next, we'll break our arrays into training/development/test arrays, each of which we save individually.

In [ ]:
N = np.shape(X)[0] - cfg["data"]["N_test"]

X_test = X[N:]
X_train_and_dev = X[0:N]
Y_test = Y[N:]
Y_train_and_dev = Y[0:N]

np.save(str(Path(processed_data_loc) / 'X_train_and_dev.npy'), X_train_and_dev)
np.save(str(Path(processed_data_loc) / 'X_test.npy'), X_test)
np.save(str(Path(processed_data_loc) / 'Y_train_and_dev.npy'), Y_train_and_dev)
np.save(str(Path(processed_data_loc) / 'Y_test.npy'), Y_test)

<span style="font-size:20px"> <b> Partitioning our data </b> </span>

When we train, we're welcome to use all of our data, but for the purposes of this notebook, we'll train on a subset of our data. We'll use 500 devices in our training and development sets combined for this example. As we discuss in our paper, a major benefit of our tandem + pretraining approach is that it lets us get away with training on a lot less data.



In [ ]:
size = 500
dev_size = int(size*0.15)
train_size = size - dev_size

train_start = 0
train_end = train_size
dev_start = train_size
dev_end = train_size + dev_size

X_train = X_train_and_dev[train_start:train_end:]
X_dev = X_train_and_dev[dev_start:dev_end:]
X_test = np.load(Path(processed_data_loc) / 'X_test.npy')

Y_train = Y_train_and_dev[train_start:train_end]
Y_dev = Y_train_and_dev[dev_start:dev_end]
Y_test = np.load(Path(processed_data_loc) / 'Y_test.npy')

Z_train = TE.concat_X_and_Y(X_train, Y_train)
Z_dev = TE.concat_X_and_Y(X_dev, Y_dev)

<span style="font-size:20px"> <b> Initalizing our neural networks </b> </span>

Now that we have our data, we're ready to begin working on the neural networks. Let's start by initializing each network. These functions automatically print the structures of each network.

In [ ]:
tf.random.set_seed(19700101)
np.random.seed(19700101)
random.seed(19700101)

model_name_forward = 'NN_forward.keras'
model_name_inverse = 'NN_inverse.keras'

model_forward = TE.build_model_forward(
    cfg["data"]["num_params"],
    cfg["data"]["num_IdVg"],
    cfg["data"]["n_points"]
)

model_inverse = TE.build_model_inverse(
    cfg["data"]["n_points"],
    cfg["data"]["num_IdVg"],
    cfg["data"]["num_feats"],
    cfg["data"]["num_params"]
)

<span style="font-size:20px"> <b> Training the forward neural network </b> </span>

We're going to begin by training the forward neural network, i.e., the neural network that approximates our physics-based solver. As shown below, the forward neural network accepts the model parameters as inputs, pushes them through a dense + gated recurrant unit (GRU) network, and estimates the resultant current-voltage curves. Here, we train this network using a built-in function in the transistor_extract module. We train using learning rate annealing and early stopping. For each annealing cycle, we train until our model performance stops improving for a specified number of cycles, reload the best model, and then continue the next annealing cycle using the best model.

<div style="text-align: center;">
  <img src="images/forward_network.png" alt="Drawing" style="width: 95%;"/>
</div>

Our loss function aims to minimize the error of the current and its derivates (with respect to V<sub>gs</sub>) in the fit in both linear and logarithmic space. See Supplementary Section 5 in our paper for the specifics of the loss function.

In [ ]:
tf.random.set_seed(19700101)
np.random.seed(19700101)
random.seed(19700101)

model_forward, _ = TE.train_forward_NN(
    working_dir,
    X_train[:, :, 0:cfg["data"]["num_IdVg"] * 2],
    Y_train,
    X_dev[:, :, 0:cfg["data"]["num_IdVg"] * 2],
    Y_dev,
    model_forward,
    model_name_forward,
    cfg["forward_model"]["lr0_forward"],
    cfg["forward_model"]["ar_forward"],
    cfg["forward_model"]["N_anneals_forward"],
    cfg["forward_model"]["patience_forward"],
    cfg["forward_model"]["bs_forward"],
    print_frequency = 100 # make this smaller to increase how frequently we
                          # print training results
)

<span style="font-size:20px"> <b> Evaluating the forward neural network </b> </span>

Now that we've trained our forward neural network, let's test it out. We'll begin by pushing the entire test set through our forward neural network and save each of the resultant fits. 

In [ ]:
# make our results folder if it doesn't exist
(Path(working_dir) / 'forward_results').mkdir(parents=True, exist_ok=True)

X_test = np.load(Path(processed_data_loc) / 'X_test.npy')[:, :, 0:cfg["data"]["num_IdVg"] * 2]
Y_test = np.load(Path(processed_data_loc) / 'Y_test.npy')

error_filename_forward = 'errors_forward.dat'

_, _, errors = TE.test_model_forward(
    X_test,
    Y_test,
    model_forward,
    Xscaling,
    Yscaling,
    TE.calc_R2,
    error_filename_forward,
    Path(working_dir) / 'forward_results',
    plot=True,
    fit_name=os.path.basename(model_name_forward).replace('.keras', ''),
    )

#subfolder_name = 'NN_forward_pred'
filenames = [str(p) for p in Path(working_dir).glob('forward_results/fits_forward/NN_forward_pred/*.dat')]
errors = []

for filename in filenames:
    error = filename.replace(str(Path(working_dir) / 'forward_results' / 'fits_forward' / 'NN_forward_pred'), '')
    error = error.replace('error=', '')
    error = error.replace('_pred.dat', '').replace('/', '')
    errors.append(float(error))

We're now ready to plot! Our test set is large -- we have 1,000 devices we can choose from -- so we suggest plotting a specific fit based on what quantile it corresponds to. We evaluate the quality of fits in terms of their coefficient of determination, R-squared. For example, if we want to see the median fit (as measured by R-squared), we should look at the 50th quantile. If we want to look at bottom 5% of all fits, we would look at the 5th quantile. Below, you can pick any quantile to examine.

In [ ]:
quantile = 5
TE.plot_forward(
                quantile/100, 
                errors, 
                str(Path(working_dir) / 'forward_results' / 'fits_forward' / 'NN_forward_pred' / 'error={:.12f}_pred.dat'),
                str(Path(working_dir) / 'forward_results' / 'fits_forward' / 'NN_forward_actual' / 'error={:.12f}_actual.dat')
                )

<span style="font-size:20px"> <b> Training the inverse neural network </b> </span>

Once we're satisified with our forward neural network, we can move onto our inverse neural network. Our inverse neural network accepts I<sub>d</sub>-V<sub>gs</sub> curves and derivatives as inputs, passes them through a dense neural network, and estimates output the model parameters that Sentaurus can use to reproduce the original current-voltage data:

<div style="text-align: center;">
  <img src="images/inverse_network.png" alt="Drawing" style="width: 95%;"/>
</div>

<span style="font-size:20px"> <b> Generating an augmented dataset for pretraining </b> </span>

Here, we use a pretraining approach to improve the initial state of our model when we train on real physics-based data. We use our trained forward neural network to generate a large augmented dataset, pretrain the inverse neural network on that, and then fine-tune on our physics based data. Let's start with generating our augmented dataset, using a built-in function.

In [ ]:
tf.random.set_seed(19700101)
np.random.seed(19700101)
random.seed(19700101)

N_augment = 100000

X_synth, Y_synth = TE.augment_data(
    model_forward,
    N_augment,
    np.shape(Y_train)[1],
    Xscaling,
    Yscaling,
    np.linspace(cfg["data"]["Vmin"], cfg["data"]["Vmax"], cfg["data"]["n_points"]),
    save=False
    )

num_synth_train = int(0.85*N_augment)
X_train_synth = X_synth[0:num_synth_train]
Y_train_synth = Y_synth[0:num_synth_train]
X_dev_synth = X_synth[num_synth_train:]
Y_dev_synth = Y_synth[num_synth_train:]
Z_train_synth = TE.concat_X_and_Y(X_train_synth, Y_train_synth)
Z_dev_synth = TE.concat_X_and_Y(X_dev_synth, Y_dev_synth)

<span style="font-size:20px"> <b> Pretraining the inverse neural network </b> </span>

Now, we use a built-in function to train our inverse neural network using this augmented dataset. Like the forward neural network, we're going to use learning rate annealing and early stopping and save only the best-performing network. 

In [ ]:
tf.random.set_seed(19700101)
np.random.seed(19700101)
random.seed(19700101)

model_inverse_pretrain, _ = TE.train_inverse_NN(
    working_dir,
    X_train_synth,
    Z_train_synth,
    X_dev_synth,
    Z_dev_synth,
    model_inverse,
    model_name_inverse,
    model_forward,
    cfg["inverse_pretraining"]["lr0_inverse_pre"],
    cfg["inverse_pretraining"]["ar_inverse_pre"],
    cfg["inverse_pretraining"]["N_anneals_inverse_pre"],
    cfg["inverse_pretraining"]["patience_inverse_pre"],
    cfg["inverse_pretraining"]["bs_inverse_pre"],
    print_frequency = 10 
    )

<span style="font-size:20px"> <b> Fine-tuning the inverse neural network </b> </span>

After pre-training, we fine-tune our inverse neural network on our original physics-based (Sentaurus) data. The training procedure is identical as before, only now we're using the original physics-based dataset. We've also updated our hyperparameters.

In [ ]:
tf.random.set_seed(19700101)
np.random.seed(19700101)
random.seed(19700101)

model_inverse_pretrain_final, _ = TE.train_inverse_NN(
    working_dir,
    X_train,
    Z_train,
    X_dev,
    Z_dev,
    model_inverse_pretrain,
    model_name_inverse,
    model_forward,
    cfg["inverse_finetuning"]["lr0_inverse_ft"],
    cfg["inverse_finetuning"]["ar_inverse_ft"],
    cfg["inverse_finetuning"]["N_anneals_inverse_ft"],
    cfg["inverse_finetuning"]["patience_inverse_ft"],
    cfg["inverse_finetuning"]["bs_inverse_ft"],
    print_frequency = 50
    )



<span style="font-size:20px"> <b> Evaluating the inverse neural network </b> </span>

We're ready to begin evaluating our trained inverse network. We'll begin by extracting out predictions for the fitting parameters and the reverse-engineered current. As we did for the forward neural network, we'll save these fits (and the extracted parameters) to file.

Ideally, we'd like to assess the quality of our fits using Sentaurus TCAD: we'd like to extract out the model parameters, input them into Sentaurus, and compare the original and predicted current. Unfortunately, this approach isn't feasible in a notebook environment, since Sentaurus TCAD is a proprietary software that takes a while to run a single simulation. Instead, we'll use a well-trained forward neural network that was trained on all 25,000 device simulations in the original test set. This network is characterized in the latter half of Supplementary Section S3 in our paper.



In [ ]:
model_forward_fully_trained = load_model(                                             
                Path(root_dir) / 'models' / 'NN_forward_well_trained.keras',                                      
                custom_objects={'CombinedMSELoss': TE.CombinedMSELoss}             
                )

In [ ]:
X_test = np.load(Path(processed_data_loc) / 'X_test.npy')
Y_test = np.load(Path(processed_data_loc) / 'Y_test.npy')
error_filename_inverse = 'errors_inverse.dat'

# make our results folder if it doesn't exist
(Path(working_dir) / 'inverse_results').mkdir(parents=True, exist_ok=True)

_, _, errors = TE.test_model_inverse_current(
    X_test,
    Y_test,
    model_forward_fully_trained,
    model_inverse,
    Xscaling,
    Yscaling,
    TE.calc_R2,
    error_filename_inverse,
    working_dir + '/inverse_results',
    deriv_error = False,
    plot = True,
    save_fits = True,
    fit_name = model_name_inverse.replace('.keras', '')
    )

filenames = [str(p) for p in Path(working_dir).glob('inverse_results/fits_inverse/pred/*.dat')]
errors = []

num_entries = len(filenames)

for filename in filenames:
    error = filename.replace(str(Path(working_dir) / 'inverse_results' / 'fits_inverse' / 'pred'), '')
    error = error.replace('error=', '')
    error = error.replace('_pred.dat', '').replace('/', '')
    errors.append(float(error))

Now we're ready to examgine the error in our fitting parameters. We use a predefined function to examine a given parameter; we can change <i>plot_idx</i> to select which fitting parameter we want to look at in more detail. Each plot_idx corresponds to a different fitting parameter:

- 0: Mobility
- 1: Schottky contact barrier height
- 2: Effective density of states
- 3: Peak donor density
- 4: Donor energy mid
- 5: Donor energy width
- 6: Peak acceptor band tail density
- 7: Acceptor band tail energy width

You will notice that some parameters are more error-prone than others. We discuss reasons for this in Supplementary Section S6 of our manuscript.

In [ ]:
plot_idx = 0
TE.plot_variables(model_inverse, plot_idx, X_test, Y_test, Xscaling, Yscaling)



Finally, we can examine the quantity of the reverse engineered fit. As before, it can be helpful to pick a specific quantile you're interested in, which helps you get a better idea of what fits across the entire test set look like. Below, choose the quantile that you want to examine -- for example, to select the 5th quantile, i.e., the 5th worst fit (as measured by the R-squared), set <i>quantile</i> to 5.



In [ ]:
quantile = 5
TE.plot_inverse(
             quantile/100, 
             errors,
             str(Path(working_dir) / 'inverse_results' / 'fits_inverse' / 'pred' / 'error={:.12f}_pred.dat'),
             str(Path(working_dir) / 'inverse_results' / 'fits_inverse' / 'actual' / 'error={:.12f}_actual.dat')
    )